In [1]:
import json
from pathlib import Path


# ============================================================
# 1. ORDNER EINSTELLEN
# ============================================================

INPUT_DIR = Path("input")
OUTPUT_DIR = Path("json_with_positions")
ERROR_FILE = Path("position_errors_2.txt")

OUTPUT_DIR.mkdir(exist_ok=True)


# ============================================================
# 2. START-/END-POSITIONEN ERMITTELN
# ============================================================

def find_annotation_positions(text, annotations):
    """
    Ermittelt für jede Annotation die start- und end-Position.

    Regeln:
    - start ist inklusive
    - end ist exklusiv
    - Mehrfach vorkommende Texte werden in der Reihenfolge
      der Annotationen zugeordnet.
    - Identische Annotationstexte direkt nacheinander können
      dieselbe Textstelle erhalten.
    """

    results = []
    errors = []

    # Speichert, welche Fundstellen eines Textes bereits
    # verwendet wurden.
    used_positions = {}

    # Die zuletzt verwendete Position einer Annotation
    # wird ebenfalls gespeichert.
    last_position = None
    last_text = None

    for index, annotation in enumerate(annotations):

        annotation_text = annotation.get("text", "")

        if not annotation_text:
            errors.append(
                f"Annotation {index}: kein Annotationstext vorhanden."
            )
            results.append(annotation)
            continue

        # ----------------------------------------------------
        # FALL 1:
        # Exakt derselbe Annotationstext wie bei der
        # unmittelbar vorhergehenden Annotation.
        #
        # Beispiel:
        #
        # Husförestånderinna -> occupation
        # Husförestånderinna -> person
        #
        # Beide beziehen sich auf dieselbe Stelle.
        # ----------------------------------------------------

        if annotation_text == last_text and last_position is not None:

            start, end = last_position

            annotation["start"] = start
            annotation["end"] = end

            results.append(annotation)
            continue

        # ----------------------------------------------------
        # FALL 2:
        # Text suchen.
        #
        # Wir suchen ab der Position der letzten Annotation,
        # damit die Annotationen grundsätzlich der Reihenfolge
        # des Originaltextes folgen.
        # ----------------------------------------------------

        search_start = 0

        if last_position is not None:
            search_start = last_position[1]

        position = text.find(annotation_text, search_start)

        # ----------------------------------------------------
        # FALL 3:
        # Falls ab der letzten Position nichts gefunden wird,
        # suchen wir noch einmal im gesamten Text.
        #
        # Das ist eine Sicherheitsmaßnahme für Annotationen,
        # die nicht streng von links nach rechts angeordnet sind.
        # ----------------------------------------------------

        if position == -1:
            position = text.find(annotation_text)

        # ----------------------------------------------------
        # FALL 4:
        # Annotation konnte nicht gefunden werden.
        # ----------------------------------------------------

        if position == -1:

            errors.append(
                f'Annotation {index}: "{annotation_text}" '
                f'konnte im Text nicht gefunden werden.'
            )

            results.append(annotation)
            continue

        # ----------------------------------------------------
        # Position berechnen
        # ----------------------------------------------------

        start = position
        end = position + len(annotation_text)

        annotation["start"] = start
        annotation["end"] = end

        # Position speichern
        last_position = (start, end)
        last_text = annotation_text

        results.append(annotation)

    return results, errors


# ============================================================
# 3. EINZELNE JSON-DATEI BEARBEITEN
# ============================================================

def process_file(input_file, output_file):

    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    text = data["text"]
    annotations = data.get("annotations", [])

    new_annotations, errors = find_annotation_positions(
        text,
        annotations
    )

    data["annotations"] = new_annotations

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )

    return errors


# ============================================================
# 4. ALLE JSON-DATEIEN VERARBEITEN
# ============================================================

all_errors = []

json_files = list(INPUT_DIR.glob("*.json"))

print(f"{len(json_files)} JSON-Dateien gefunden.\n")

for input_file in json_files:

    output_file = OUTPUT_DIR / input_file.name

    try:

        errors = process_file(
            input_file,
            output_file
        )

        if errors:

            print(f"⚠ {input_file.name}")

            for error in errors:
                print("   ", error)

            all_errors.append(
                (input_file.name, errors)
            )

        else:

            print(f"✓ {input_file.name}")

    except Exception as e:

        print(f"❌ Fehler bei {input_file.name}: {e}")

        all_errors.append(
            (input_file.name, [str(e)])
        )


# ============================================================
# 5. FEHLERBERICHT SPEICHERN
# ============================================================

with open(ERROR_FILE, "w", encoding="utf-8") as f:

    if not all_errors:

        f.write(
            "Keine Fehler gefunden.\n"
        )

    else:

        for filename, errors in all_errors:

            f.write(f"{filename}\n")
            f.write("-" * len(filename) + "\n")

            for error in errors:
                f.write(f"- {error}\n")

            f.write("\n")


# ============================================================
# 6. ABSCHLUSS
# ============================================================

print("\n--------------------------------")
print("Verarbeitung abgeschlossen.")
print(f"Dateien insgesamt: {len(json_files)}")
print(f"Dateien mit Fehlern: {len(all_errors)}")
print(f"Fehlerbericht: {ERROR_FILE}")
print(f"Ausgabeordner: {OUTPUT_DIR}")
print("--------------------------------")

4 JSON-Dateien gefunden.

✓ 1906_09_15_SD_LP_001.json
✓ 1906_09_16_SD_LP_003.json
✓ 1906_09_19_SD_LP_001.json
✓ 1906_09_23_SD_LP_003.json

--------------------------------
Verarbeitung abgeschlossen.
Dateien insgesamt: 4
Dateien mit Fehlern: 0
Fehlerbericht: position_errors_2.txt
Ausgabeordner: json_with_positions
--------------------------------
